### Tools

In [ ]:
import os
# from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

model = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",model_provider="openrouter") # need to add ,model_provider="openrouter", then only it works

response = model.invoke("What is the distance between Earth and Mars?")
print(response.content)

The distance between Earth and Mars **changes constantly** because both planets orbit the Sun at different speeds and distances.

Here are the key figures:

| Scenario | Distance (km) | Distance (miles) | Light Travel Time |
| :--- | :--- | :--- | :--- |
| **Closest Possible (Theoretical Opposition at Perihelion)** | **~54.6 million km** | **~33.9 million miles** | ~3 minutes |
| **Average Distance** | **~225 million km** | **~140 million miles** | ~12.5 minutes |
| **Farthest Possible (Sun between them)** | **~401 million km** | **~249 million miles** | ~22 minutes |

### Why does it vary so much?
1.  **Elliptical Orbits:** Neither planet orbits in a perfect circle. Mars has a significantly elliptical orbit.
2.  **Different Orbital Speeds:** Earth orbits faster (1 year) than Mars (1.88 years).
3.  **"Opposition" vs. "Conjunction":**
    *   **Opposition (Closest):** Earth passes directly between Mars and the Sun. This happens every **~26 months**. Because of elliptical orbits, some op

In [ ]:
from langchain.tools import tool

@tool #decorator to register the function as a tool in langchain
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is cloudy with a temperature."

model_with_tool = model.bind_tool([get_weather]) #bind the tool to the model

In [29]:
# when validation fails, we need to run above code again to rebind the tool to the model, otherwise it will not work
response = model_with_tool.invoke("What is the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool call: {tool_call['name']}")
    print(f"Args:{tool_call['args']}")

    # print(f"Tool called: {tool_call.input}") #does not work as tool_call is a dict, not an object
    # print(f"Tool output: {tool_call.output}") #does not work as tool_call is a dict, not an object

content='' additional_kwargs={'reasoning_content': 'The user is asking for the weather in New York. I need to use the get_weather tool with location "New York". Let\'s call the function.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for the weather in New York. I need to use the get_weather tool with location "New York". Let\'s call the function.'}]} response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790828673-yuWfRbUvL4Fq3baFlglm', 'created': 1790828673, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}} id='lc_run--01a0f5b4-f915-7972-b1ae-9a98d1584d33-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call-65af4c04-6cc5-4711-a8b2-f8584bf82fff', 'type': '

### Tool Execution Loop

In [ ]:
# Step 1 model genrated the tool call
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2 execute the tools and collect the results
for tool_call in ai_msg.tool_calls:
   # Execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call) # to get the result of the tool call
    messages.append(tool_result)

# Step 3 send the tool results back to the model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.content)
messages

The current weather in New York is cloudy with a temperature.


[{'role': 'user', 'content': 'What is the weather in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for the weather in New York. I need to use the get_weather tool with location "New York". Let\'s call the function.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for the weather in New York. I need to use the get_weather tool with location "New York". Let\'s call the function.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1790829112-ubvZ6bM2zTrTBGifvujr', 'created': 1790829112, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a0f5bb-af31-7252-95f5-a55a02a371a6-0', tool_calls=[{'name': 'get_weather', 'args': {'loc